In [1]:
import sys
sys.path.append("../src")

In [2]:
from pyspark.sql import SparkSession
from generate_data import generate_logistics_data
from features import clean_bronze_to_silver, build_gold_features
from train_model import train_and_log
import os

In [3]:
#!pip install mlflow

In [4]:
#!pip install pyspark
# PySpark requires a Java Development Kit (JDK) installed on your Mac to run locally, 
# because Apache Spark runs on the Java Virtual Machine (JVM) under the hood.
# conda install -c conda-forge openjdk=17

In [5]:
# java -XshowSettings:properties -version 2>&1 | grep java.home
# Your JAVA_HOME is /opt/anaconda3/lib/jvm.

In [6]:
os.environ["JAVA_HOME"] = "/opt/anaconda3/lib/jvm"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-java-options='--add-opens=java.base/sun.nio.ch=ALL-UNNAMED' pyspark-shell"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("LogisticsDelayML") \
    .getOrCreate()

print("Spark initialized:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/23 14:59:37 WARN Utils: Your hostname, yangs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 10.78.181.32 instead (on interface en0)
26/11/23 14:59:37 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/23 14:59:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark initialized: 4.3.0


In [7]:
!pwd

/Users/yliu/Desktop/learning/logistics-delay-ml-pipeline/notebooks


In [8]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("LogisticsDelayML") \
    .getOrCreate()

print("Spark initialized:", spark.version)
# Spark 4.3.0 is officially up and running on your Mac!

Spark initialized: 4.3.0


In [9]:
# load the generated raw data
bronze_df = spark.read.csv("../data/bronze_shipments.csv", header = True, inferSchema = True)
bronze_df.printSchema()
bronze_df.show(5)

root
 |-- shipment_id: string (nullable = true)
 |-- dispatch_timestamp: timestamp (nullable = true)
 |-- origin_region: string (nullable = true)
 |-- dest_region: string (nullable = true)
 |-- carrier: string (nullable = true)
 |-- weather: string (nullable = true)
 |-- distance_km: double (nullable = true)
 |-- weight_kg: double (nullable = true)
 |-- expected_hours: double (nullable = true)
 |-- actual_hours: double (nullable = true)
 |-- is_delayed: integer (nullable = true)

+-----------+-------------------+-------------+-----------+------------+-------+-----------+---------+--------------+------------+----------+
|shipment_id| dispatch_timestamp|origin_region|dest_region|     carrier|weather|distance_km|weight_kg|expected_hours|actual_hours|is_delayed|
+-----------+-------------------+-------------+-----------+------------+-------+-----------+---------+--------------+------------+----------+
| SHP-100000|2026-02-05 20:00:00|         East|      North|  PrimeCargo|  Clear|     1753

In [10]:
#clean_bronze_to_silver, build_gold_features

In [11]:
silver_df = clean_bronze_to_silver(bronze_df)
silver_df.show(5,vertical=True)

-RECORD 0---------------------------------
 shipment_id        | SHP-100000          
 dispatch_timestamp | 2026-02-05 20:00:00 
 origin_region      | East                
 dest_region        | North               
 carrier            | PrimeCargo          
 weather            | Clear               
 distance_km        | 1753.8              
 weight_kg          | 4233.6              
 expected_hours     | 38.98               
 actual_hours       | 39.34               
 is_delayed         | 0                   
-RECORD 1---------------------------------
 shipment_id        | SHP-100001          
 dispatch_timestamp | 2026-06-07 04:00:00 
 origin_region      | West                
 dest_region        | South               
 carrier            | SwiftFreight        
 weather            | Clear               
 distance_km        | 234.8               
 weight_kg          | 4776.6              
 expected_hours     | 15.61               
 actual_hours       | 15.97               
 is_delayed

In [12]:
gold_df = build_gold_features(silver_df)
gold_df.show(5, truncate=False)

+-----------+-------------------+-------------+-----------+------------+-------+-----------+---------+--------------+------------+----------+-------------+------------------+----------+---------------+-------------+-----------------+
|shipment_id|dispatch_timestamp |origin_region|dest_region|carrier     |weather|distance_km|weight_kg|expected_hours|actual_hours|is_delayed|dispatch_hour|dispatch_dayofweek|is_weekend|is_cross_region|weight_per_km|is_severe_weather|
+-----------+-------------------+-------------+-----------+------------+-------+-----------+---------+--------------+------------+----------+-------------+------------------+----------+---------------+-------------+-----------------+
|SHP-100000 |2026-02-05 20:00:00|East         |North      |PrimeCargo  |Clear  |1753.8     |4233.6   |38.98         |39.34       |0         |20           |5                 |0         |1              |2.414        |0                |
|SHP-100001 |2026-06-07 04:00:00|West         |South      |Swift

In [13]:
gold_pdf = gold_df.toPandas()
pipeline, metrics = train_and_log(gold_pdf, experiment_name="logistics-delay-ml")

Run logged successfully. Metrics:{'accuracy': 0.84, 'f1_score': 0.7142857142857143, 'roc_auc': 0.91452161243322}


In [ ]:
# View Your MLflow UI Locally
# To inspect the logged parameters, metrics (accuracy, f1_score, roc_auc), 
# and model artifact in your browser, open a terminal in your project root and run:
# mlflow ui
# Then open http://127.0.0.1:5000 in your browser.